In [1]:
from dataclasses import dataclass
from typing import Callable

In [2]:
@dataclass
class DiffResults:
    differences: [[int]]
    ks: [int]
    xs: [[int]]

In [3]:
@dataclass
class Array:
    dom: int
    cod: int
    f: Callable[[int], int]

In [4]:
@dataclass
class InducedArray:
    dom: int
    rel_dom: int
    cod: int
    rel_cod: int
    f: Callable[[int], int]

In [5]:
@dataclass
class Embedding:
    array: Array
    induced_array: InducedArray
    iota_1: Callable[[int], int]
    iota_2: Callable[[int], int]
    in_dom_image: Callable[[int], bool]
    in_cod_image: Callable[[int], bool]

In [6]:
def gen_contiguous_embedding(array: Array, rel_dom: int, rel_cod: int) -> Embedding:
    n = array.dom

    # --- Domain embedding ---
    if rel_dom <= 0:
        # ℤ case
        def iota_1(i: int) -> int:
            if not (1 <= i <= n):
                raise ValueError("i out of domain")
            return i

        def in_dom_image(g: int) -> bool:
            return 1 <= g <= n

        def inv_iota_1(g: int) -> int:
            return g

    else:
        # Z_{rel_dom} case (0-based)
        def iota_1(i: int) -> int:
            if not (1 <= i <= n):
                raise ValueError("i out of domain")
            return i - 1

        def in_dom_image(g: int) -> bool:
            return 0 <= g < n

        def inv_iota_1(g: int) -> int:
            return g + 1

    # --- Codomain embedding ---
    if rel_cod <= 0:
        def iota_2(i: int) -> int:
            if not (1 <= i <= n):
                raise ValueError("i out of codomain")
            return i

        def in_cod_image(h: int) -> bool:
            return 1 <= h <= n

    else:
        def iota_2(i: int) -> int:
            if not (1 <= i <= n):
                raise ValueError("i out of codomain")
            return i - 1

        def in_cod_image(h: int) -> bool:
            return 0 <= h < n

    # --- Induced map ---
    def induced_f(g: int) -> int:
        if not in_dom_image(g):
            raise ValueError(f"{g} not in domain image")

        x = inv_iota_1(g)
        y = array.f(x)
        h = iota_2(y)

        if not in_cod_image(h):
            raise ValueError(f"{h} not in codomain image")

        return h

    induced = InducedArray(
        dom=n,
        rel_dom=rel_dom,
        cod=array.cod,
        rel_cod=rel_cod,
        f=induced_f
    )

    return Embedding(
        array=array,
        induced_array=induced,
        iota_1=iota_1,
        iota_2=iota_2,
        in_dom_image=in_dom_image,
        in_cod_image=in_cod_image
    )

In [7]:
def calculate_differences(embedding: Embedding) -> DiffResults:
    induced = embedding.induced_array
    rel_dom = induced.rel_dom
    rel_cod = induced.rel_cod
    f = induced.f

    in_dom = embedding.in_dom_image

    res = DiffResults([], [], [])

    def add_G(g: int, k: int) -> int:
        if rel_dom <= 0:
            return g + k
        return (g + k) % rel_dom

    k_upper = induced.dom + 1 if rel_dom <= 0 else rel_dom
    g_upper = induced.dom if rel_dom <= 0 else rel_dom # might be bug

    for k in range(1, k_upper):
        res.ks.append(k)
        res.xs.append([])
        res.differences.append([])

        for g in range(1 if rel_dom <= 0 else 0, g_upper):
            if not in_dom(g):
                continue

            gk = add_G(g, k)

            if not in_dom(gk):
                continue

            res.xs[k - 1].append(g)

            if rel_cod <= 0:
                diff = f(gk) - f(g)
            else:
                diff = (f(gk) - f(g)) % rel_cod

            res.differences[k - 1].append(diff)

    return res

In [8]:
def distinct_differences(diff: DiffResults) -> bool:
    for lst in diff.differences:
        seen = set()
        for d in lst:
            if d in seen:
                return False
            else:
                seen.add(d)
    return True

In [9]:
welch2431 = Array(4, 4, lambda i: 2^i % 5)

standard_emb = gen_contiguous_embedding(welch2431, 0, 0)
circular1_emb = gen_contiguous_embedding(welch2431, 4, 5)
circular2_emb = gen_contiguous_embedding(welch2431, 5, 5)

standard = calculate_differences(standard_emb)
circular1 = calculate_differences(circular1_emb)
circular2 = calculate_differences(circular2_emb)

In [10]:
print(", ".join(f"{welch2431.f(i)}" for i in range(1, 5)))
print(standard, distinct_differences(standard))
print(circular1, distinct_differences(circular1))
print(circular2, distinct_differences(circular2))

2, 4, 3, 1
DiffResults(differences=[[2, -1, -2], [1, -3], [-1], []], ks=[1, 2, 3, 4], xs=[[1, 2, 3], [1, 2], [1], []]) True
DiffResults(differences=[[2, 4, 3, 1], [1, 2, 4, 3], [4, 3, 1, 2]], ks=[1, 2, 3], xs=[[0, 1, 2, 3], [0, 1, 2, 3], [0, 1, 2, 3]]) True
DiffResults(differences=[[2, 4, 3], [1, 2, 1], [4, 4, 3], [3, 1, 2]], ks=[1, 2, 3, 4], xs=[[0, 1, 2], [0, 1, 3], [0, 2, 3], [1, 2, 3]]) False


In [31]:
def generate_welch_arrays(n_primes: int) -> [Array]:
    primes = (p for p in Primes() if p > 2)
    res = []
    for _ in range(n_primes):
        p = next(primes)
        b = Integers(p)(primitive_root(p))
        prim_roots = [b^k for k in range(p-1) if gcd(k, p-1) == 1]
        for a in prim_roots:
            res.append(Array(p-1, p-1, lambda i: (a^i) % p))
    return res

In [32]:
(p for p in Primes() if p > 2)

<generator object <genexpr> at 0x7f364e8365e0>

In [33]:
welch_arrays = generate_welch_arrays(5)

In [34]:
for w in welch_arrays:
    print(", ".join(f"{w.f(i)}" for i in range(1, w.dom + 1)))

7, 10
7, 10, 5, 9
7, 10, 5, 9
7, 10, 5, 9, 11, 12
7, 10, 5, 9, 11, 12
7, 10, 5, 9, 11, 12, 6, 3, 8, 4
7, 10, 5, 9, 11, 12, 6, 3, 8, 4
7, 10, 5, 9, 11, 12, 6, 3, 8, 4
7, 10, 5, 9, 11, 12, 6, 3, 8, 4
7, 10, 5, 9, 11, 12, 6, 3, 8, 4, 2, 1
7, 10, 5, 9, 11, 12, 6, 3, 8, 4, 2, 1
7, 10, 5, 9, 11, 12, 6, 3, 8, 4, 2, 1
7, 10, 5, 9, 11, 12, 6, 3, 8, 4, 2, 1


In [14]:
for w in welch_arrays:
    emb = gen_contiguous_embedding(w, w.dom + 1, w.cod + 1)
    diffs = calculate_differences(emb)
    print(distinct_differences(diffs))

True
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False

In [15]:
def generate_all_1_1_embeddings(array: Array) -> [Embedding]:
    embs = []
    for a in range(array.dom + 1):
        for b in range(array.cod + 1):
            
    return embs

IndentationError: expected an indented block after 'for' statement on line 4 (199567816.py, line 6)